# Create datasets for models
This notebook is used to create analytical sets for clinical data analysis, specifically as input for logistic regression and Cox proportional hazard models.

* Author: Lietsel Jones
* Last modified: August 10th, 2026

In [ ]:
# Mount bucket
! gcsfuse --dir-mode 777 --file-mode 777 --implicit-dirs {BUCKET_NAME} ~/{WORKING_DIR}/

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter
import statsmodels.api as sm

from datetime import datetime
# Get the current date in YYYY-MM-DD format
today = datetime.now().strftime('%Y-%m-%d')

In [ ]:
# Load the analysis dataset
df_analysis = pd.read_csv(f'/{WORKING_DIR}/temp/hy_df_analysis_2026-08-10.csv', low_memory=False)

df_analysis.head()

### Dataset for logistic regression

In [ ]:
# If hoehn_and_yahr_stage is greater than or equal to 3, then event = 1, else event = 0
dfc = df_analysis[df_analysis['study_days'] == 0].copy()
dfc["event"] = np.where(dfc["HY"] >= 3, 1, 0)

In [ ]:
# Grand-mean-centered age and disease duration (+ their squares).
# Centering is done once on the
# full pooled dataset, then applied to every cohort, so all cohorts share the same
# reference point in the meta-analysis.
grand_mean_age = dfc['age_at_baseline'].mean()
grand_mean_dd = dfc['disease_duration'].mean()

dfc['c_age'] = dfc['age_at_baseline'] - grand_mean_age
dfc['c_age_sq'] = dfc['c_age'] ** 2
dfc['c_disease_duration'] = dfc['disease_duration'] - grand_mean_dd
dfc['c_dd_sq'] = dfc['c_disease_duration'] ** 2

print(f'Grand mean age at baseline: {grand_mean_age:.4f}')
print(f'Grand mean disease duration: {grand_mean_dd:.4f}')

In [ ]:
cohort_summary = dfc.groupby('cohort').agg(
    n=('disease_duration', 'size'),
    mean_duration=('disease_duration', 'mean'),
    median_duration=('disease_duration', 'median'),
    sd_duration=('disease_duration', 'std'),
    pct_HY_le2=('HY', lambda x: (x <= 2).mean()),
    pct_de_novo_proxy=('disease_duration', lambda x: (x <= 2).mean())
).sort_values('mean_duration')

cohort_summary

In [ ]:
# Preview final cross-sectional dataset
print(dfc.shape)
print(dfc.columns)
dfc.head()

In [ ]:
# export final cross-sectional dataset for analysis with date of export in filename
dfc.to_csv(f"./{WORKING_DIR}/datasets/clin_hy_cs_dataset_for_LR_{today}.csv", index=False)

### Dataset for Cox proportional hazards

In [ ]:
df_lt = df_analysis[df_analysis['lt'] == 1].copy()
print(df_lt.shape)
print(df_lt['cohort'].value_counts())

Helper function to create survival dataset.

In [ ]:
def create_survival_df(
    df,
    thres,
    direction,
    outcome,
    id_col='id',
    time_col='study_days',
    baseline_cols=None,
    verbose=True
):
    """
    Create one-row-per-person survival dataset from longitudinal data
    """

    if baseline_cols is None:
        baseline_cols = [
            'cohort', 
            'age_at_baseline', 'age_at_onset', 
            'disease_duration', 'sex',
             'hy1_baseline'
        ]

    df = df.copy()

    # Define event
    if direction == 'greater':
        df['event'] = (df[outcome] >= thres).astype(int)
    elif direction == 'less':
        df['event'] = (df[outcome] <= thres).astype(int)
    else:
        raise ValueError('direction must be "greater" or "less"')

    # Sort by GP2ID and time col
    df = df.sort_values([id_col, time_col])

    # Exclude individuals who already met the event threshold at their baseline (first) visit
    baseline_event_ids = (
        df.groupby(id_col, as_index=False)
          .first()
          .loc[lambda x: x['event'] == 1, id_col]
    )
    n_excluded_baseline = baseline_event_ids.nunique()
    df = df[~df[id_col].isin(baseline_event_ids)]

    # Baseline covars
    df_baseline = (
        df[[id_col] + baseline_cols]
        .groupby(id_col, as_index=False)
        .first()
    )

    # Flag time-varying baseline vars
    baseline_nunique = (
        df.groupby(id_col)[baseline_cols]
          .nunique(dropna=False)
    )
    baseline_conflicts = baseline_nunique[baseline_nunique.gt(1).any(axis=1)]

    # Event timing for first occurrence
    df_event = (
        df[df['event'] == 1]
        .groupby(id_col, as_index=False)
        .first()[[id_col, time_col]]
        .rename(columns={time_col: 'visit_month_event'})
    )

    # Follow-ups
    df_sv = (
        df.groupby(id_col)[time_col]
          .agg(
              visit_month_first='min',
              visit_month_last='max',
              n_obs='count'
          )
          .reset_index()
    )

    df_sv = df_sv.merge(df_event, on=id_col, how='left')
    df_sv['event'] = df_sv['visit_month_event'].notna().astype(int)

    df_sv['visit_month_censored'] = np.where(
        df_sv['event'] == 1,
        df_sv['visit_month_event'],
        df_sv['visit_month_last']
    )

    df_sv['censored_month'] = (
        df_sv['visit_month_censored'] - df_sv['visit_month_first']
    )
    df_sv['follow_up_month'] = (
        df_sv['visit_month_last'] - df_sv['visit_month_first']
    )

    # Outcome at first and last visit
    df_scores = (
        df.groupby(id_col)[outcome]
          .agg(score_first='first', score_last='last')
          .reset_index()
    )

    df_sv = df_sv.merge(df_scores, on=id_col, how='left')

    # Final survival dataset
    df_sv_return = df_baseline.merge(
        df_sv[
            [
                id_col, 'n_obs', 'follow_up_month',
                'visit_month_first', 'visit_month_last',
                'score_first', 'score_last',
                'event', 'censored_month'
            ]
        ],
        on=id_col,
        how='left'
    )

    # Safety checks
    if not df_sv_return[id_col].is_unique:
        raise RuntimeError("Duplicate GP2IDs detected in survival dataset")

    # QC summary
    if verbose:
        qc = {
            'n_participants': df_sv_return.shape[0],
            'n_excluded_event_at_baseline': int(n_excluded_baseline),
            'n_events': int(df_sv_return['event'].sum()),
            'event_rate': df_sv_return['event'].mean(),
            'median_follow_up_months': df_sv_return['follow_up_month'].median(),
            'max_follow_up_months': df_sv_return['follow_up_month'].max(),
            'baseline_conflicts_n': baseline_conflicts.shape[0],
        }

        print("Survival dataset QC summary")
        for k, v in qc.items():
            print(f"  {k}: {v}")

        if baseline_conflicts.shape[0] > 0:
            print("\n Baseline variables varying within participant:")
            print(baseline_conflicts.head())

    return df_sv_return


In [ ]:
def plot_km_curve(df_sv, strata, label, path=None):
    kmf = KaplanMeierFitter()
    plt.figure(figsize=(10, 6))
    
    ax = plt.subplot(111)
    
    # Plot the KM curve for each group in strata
    for name, grouped_df in df_sv.groupby(strata):
        kmf.fit(durations=grouped_df['censored_month'], event_observed=grouped_df['event'], label=name)
        kmf.plot_survival_function(ax=ax, ci_show=False)
    
    # Manually adding at-risk numbers
    event_tables = {}
    for name, grouped_df in df_sv.groupby(strata):
        kmf.fit(durations=grouped_df['censored_month'], event_observed=grouped_df['event'], label=name)
        event_tables[name] = kmf.event_table['at_risk']
    
    plt.title(f'Kaplan-Meier Survival Curve: Event = {label}')
    plt.xlabel('Time (Months)')
    plt.ylabel('Survival Probability')
    plt.ylim([0,1])
    plt.xlim([0,120])
    plt.legend(title=strata)
    plt.grid(True)
    if path:
        plt.savefig(path)
    else:
        plt.show()

In [ ]:
# Survival dataset - HY >= 3
df_surv = create_survival_df(df_lt, 3, 'greater', 'HY')

# Plot KM curve
plot_km_curve(df_surv, 'cohort', '3 or greater')

In [ ]:
df = df_surv.copy()

In [ ]:
# Add study_days column: outcome.censored_month - outcome.visit_month_first
df_surv["study_days"] = df_surv["censored_month"] - df_surv["visit_month_first"]

# Recode sex values
df_surv["sex"] = df_surv["sex"].replace({"Male": 1, "Female": 2})
df_surv = df_surv[["id", "cohort", "age_at_baseline", "sex", "disease_duration", "event", "study_days", "hy1_baseline"]]

df_surv.head()

In [ ]:
# Export final survival dataset for Cox proportional hazards models
df_surv.to_csv(f"./{WORKING_DIR}/datasets/clin_surv_dataset_for_CPH_{today}.csv", index=False)